# Dopamine-Modulated Spiking Neural Network for HalfCheetah MuJoCo Control

## Key Features
- **Dopamine baseline** (reward prediction error) learning
- **Asymmetric plasticity**: fast LTP, slow LTD
- **Neurogenesis**: replaces inactive neurons with fresh weights
- **Homeostatic mechanisms**: adaptive threshold, synaptic scaling, weight decay
- **Continuous control**: maps observations to actions for HalfCheetah
- **Separate input channels**: observations and reward go to different neurons

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import gymnasium as gym
import mujoco
from collections import deque

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("Gymnasium version:", gym.__version__)
print("MuJoCo version:", mujoco.__version__)

I0000 00:00:1788322532.119939    2918 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788322532.155279    2918 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788322532.883386    2918 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0
Gymnasium version: 1.3.0
MuJoCo version: 3.12.0


## Network Configuration for HalfCheetah

In [2]:
# Network architecture
N_TOTAL = 200
N_EXCITATORY = 160
N_INHIBITORY = 40             # 20% inhibitory

EXCITATORY_IDS = list(range(0, 160))
INHIBITORY_IDS = list(range(160, 200))

# Ant-v4 specific I/O
OBSERVATION_DIM = 105            # Ant-v4 has 105 observations (not 27!)
ACTION_DIM = 8                   # Still 8 actions
               # HalfCheetah action space

# Input neurons for observations (17 observation values → 17 input neurons)
OBSERVATION_INPUT_IDS = list(range(0, 105))   # Neurons 0-104
REWARD_INPUT_ID = [105]                      # Neuron 17 for reward signal

# Output neurons for actions (6 action values → multiple neurons per action)
NEURONS_PER_ACTION = 4
ACTION_OUTPUT_IDS = []
for action_idx in range(ACTION_DIM):
    start = 106 + action_idx * NEURONS_PER_ACTION
    ACTION_OUTPUT_IDS.extend(list(range(start, start + NEURONS_PER_ACTION)))

# Remaining neurons (36-63) for internal processing
FREE_NEURONS = list(range(138, 200))
# Output reading
OUTPUT_IDS = ACTION_OUTPUT_IDS
# Input scaling for HalfCheetah
OBSERVATION_SCALING = 1        # Adjust if observations are different range
REWARD_SCALING = 0.5
        # Reward scale (HalfCheetah rewards can be large)

# Action decoding parameters
ACTION_DECODING_MODE = "rate"    # "rate" = firing rate, "potential" = membrane potential
ACTION_SCALING = 1.0             # NEW (Ant actions need less amplification)
ACTION_WINDOW = 15               # Steps to average for rate-based decoding

# Neuron dynamics
V_THRESHOLD_EXC = 1.0
V_THRESHOLD_INH = 0.7
V_RESET = 0.0

# Learning parameters
LEARNING_RATE_LTP = 0.005       # Lower for larger network
LEARNING_RATE_LTD = 0.00075     # Proportional
ELIGIBILITY_DECAY = 0.95
WEIGHT_MAX = 5.0
WEIGHT_MIN_EXC = 0.0
WEIGHT_MIN_INH = -5.0

# Dopamine baseline parameters
BASELINE_LEARNING_RATE = 0.05
INITIAL_BASELINE = 0.0

# Homeostatic mechanisms
TARGET_FIRING_RATE = 0.1
SYNAPTIC_SCALING_INTERVAL = 100
ADAPTIVE_THRESHOLD_RISE = 0.3
ADAPTIVE_THRESHOLD_DECAY = 0.95
WEIGHT_DECAY_RATE = 0.999

# Neurogenesis parameters
NEUROGENESIS_INTERVAL = 1000     # Longer for larger network
NEURONS_TO_REPLACE = 30          # 15% of 200
ACTIVITY_THRESHOLD = 0.05       # Lower for larger network
EXCLUDE_FROM_REPLACEMENT = OBSERVATION_INPUT_IDS + REWARD_INPUT_ID + ACTION_OUTPUT_IDS
# Internal simulation steps per environment step
INTERNAL_STEPS = 15              # Run network 10 times per MuJoCo step

print(f"Network: {N_TOTAL} neurons ({N_EXCITATORY} E, {N_INHIBITORY} I)")
print(f"Observation inputs: {OBSERVATION_INPUT_IDS}")
print(f"Reward input: {REWARD_INPUT_ID}")
print(f"Action outputs: {ACTION_OUTPUT_IDS}")
print(f"Free processing neurons: {FREE_NEURONS}")
print(f"Internal steps per env step: {INTERNAL_STEPS}")
print(f"Action decoding: {ACTION_DECODING_MODE} mode")

Network: 200 neurons (160 E, 40 I)
Observation inputs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104]
Reward input: [105]
Action outputs: [106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137]
Free processing neurons: [138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 1

## Initialize Network Class

In [3]:
# Run this cell FIRST after restarting kernel

import tensorflow as tf
import numpy as np
from collections import deque

class SpikingNeuralNetwork:
    def __init__(self):
        self.n_total = N_TOTAL
        
        # Neuron type masks
        self.is_excitatory = tf.constant(
            [1.0 if i in EXCITATORY_IDS else 0.0 for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        self.is_inhibitory = tf.constant(
            [1.0 if i in INHIBITORY_IDS else 0.0 for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        
        # Base thresholds
        self.base_threshold = tf.constant(
            [V_THRESHOLD_EXC if i in EXCITATORY_IDS else V_THRESHOLD_INH 
             for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        self.scaling_spike_counter = tf.Variable(
            tf.zeros(N_TOTAL), dtype=tf.float32, trainable=False
        )
        
        self.neurogenesis_spike_counter = tf.Variable(
            tf.zeros(N_TOTAL), dtype=tf.float32, trainable=False
        )
        
        self.scaling_steps = tf.Variable(
            0, dtype=tf.int32, trainable=False
        )
        
        self.neurogenesis_steps = tf.Variable(
            0, dtype=tf.int32, trainable=False
        )
        # Weight matrix
        initializer = tf.keras.initializers.GlorotUniform()
        W_raw = initializer(shape=(N_TOTAL, N_TOTAL))
        W_init = self._apply_weight_constraints(W_raw)
        self.W = tf.Variable(W_init, dtype=tf.float32, name='weights')
        
        # Membrane potentials
        self.V = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='potential')
        
        # Adaptive thresholds
        self.adaptive_threshold = tf.Variable(
            self.base_threshold, dtype=tf.float32, name='adaptive_threshold'
        )
        
        # Spike states
        self.spikes = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='spikes')
        self.prev_spikes = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='prev_spikes')
        
        # Eligibility traces
        self.eligibility = tf.Variable(
            tf.zeros((N_TOTAL, N_TOTAL)), dtype=tf.float32, name='eligibility'
        )
        
        # Bias
        bias_init = tf.concat([
            tf.random.uniform((N_EXCITATORY,), -0.01, 0.01),
            tf.random.uniform((N_INHIBITORY,), -0.1, -0.02)
        ], axis=0)
        self.bias = tf.Variable(bias_init, dtype=tf.float32, name='bias')
        
        # ===== ALL REQUIRED ATTRIBUTES =====
        self.baseline = tf.Variable(INITIAL_BASELINE, dtype=tf.float32, name='baseline')
        self.spike_counter = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32)
        self.steps_since_scaling = tf.Variable(0, dtype=tf.int32)
        self.steps_since_neurogenesis = tf.Variable(0, dtype=tf.int32)
        self.neurons_replaced_count = tf.Variable(0, dtype=tf.int32)
        self.replacement_history = []
        self.spike_history_buffer = deque(maxlen=ACTION_WINDOW)
        self.total_reward = tf.Variable(0.0, dtype=tf.float32)
        self.total_dopamine = tf.Variable(0.0, dtype=tf.float32)
        
        self.activity_steps = tf.Variable(
        0,
        dtype=tf.int32,
        trainable=False)

        
    def _apply_weight_constraints(self, W):
        W_np = W.numpy() if hasattr(W, 'numpy') else W
        for j in range(N_TOTAL):
            if j in EXCITATORY_IDS:
                W_np[:, j] = np.maximum(W_np[:, j], 0.0)
            elif j in INHIBITORY_IDS:
                W_np[:, j] = np.minimum(W_np[:, j], 0.0)
        np.fill_diagonal(W_np, 0.0)
        return tf.constant(W_np, dtype=tf.float32)
    
    def reset_state(self):
        self.V.assign(tf.zeros(N_TOTAL))
        self.adaptive_threshold.assign(self.base_threshold)
        self.spikes.assign(tf.zeros(N_TOTAL))
        self.prev_spikes.assign(tf.zeros(N_TOTAL))
        self.eligibility.assign(tf.zeros((N_TOTAL, N_TOTAL)))
        self.spike_counter.assign(tf.zeros(N_TOTAL))
        self.steps_since_scaling.assign(0)
        self.steps_since_neurogenesis.assign(0)
        self.spike_history_buffer.clear()
        self.total_reward.assign(0.0)
        self.total_dopamine.assign(0.0)

# Create network
network = SpikingNeuralNetwork()
print("Network initialized!")
print(f"Weight matrix shape: {network.W.shape}")
print(f"Baseline: {network.baseline.numpy():.3f}")

Network initialized!
Weight matrix shape: (200, 200)
Baseline: 0.000


E0000 00:00:1788322533.449737    2918 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## visu

In [4]:
import tkinter as tk
from tkinter import ttk
import numpy as np
from PIL import Image, ImageTk
import threading
import time

class TkNetworkVisualizer:
    def __init__(self, network, history_len=200, update_interval_ms=100):
        """
        network: your SpikingNeuralNetwork instance
        history_len: number of recent spikes to keep in raster
        update_interval_ms: how often to refresh the GUI (milliseconds)
        """
        self.net = network
        self.history_len = history_len
        self.update_interval = update_interval_ms
        
        # Data storage
        self.spike_history = []          # list of (time_step, neuron_id)
        self.step_count = 0
        
        # State
        self.paused = False
        self.running = True
        
        # Build the Tk window
        self.root = tk.Tk()
        self.root.title("Spiking Neural Network - Live Monitor")
        self.root.geometry("1000x700")
        
        # Create frames
        top_frame = tk.Frame(self.root)
        top_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        
        left_frame = tk.Frame(top_frame)
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        right_frame = tk.Frame(top_frame)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # ---- Neuron grid (Canvas) ----
        self.grid_canvas = tk.Canvas(left_frame, width=500, height=500, bg='black')
        self.grid_canvas.pack(fill=tk.BOTH, expand=True)
        
        # ---- Weight heatmap (Label with image) ----
        self.heatmap_label = tk.Label(right_frame, text="Weight Heatmap", bg='white')
        self.heatmap_label.pack(fill=tk.BOTH, expand=True)
        
        # ---- Raster (Canvas) ----
        self.raster_canvas = tk.Canvas(self.root, width=900, height=150, bg='white')
        self.raster_canvas.pack(side=tk.BOTTOM, fill=tk.X, padx=10, pady=10)
        
        # ---- Controls ----
        control_frame = tk.Frame(self.root)
        control_frame.pack(side=tk.BOTTOM, fill=tk.X, pady=5)
        
        self.pause_btn = tk.Button(control_frame, text="Pause", command=self.toggle_pause)
        self.pause_btn.pack(side=tk.LEFT, padx=5)
        
        self.speed_scale = tk.Scale(control_frame, from_=0.1, to=5.0, resolution=0.1,
                                    orient=tk.HORIZONTAL, label="Speed", length=150)
        self.speed_scale.set(1.0)
        self.speed_scale.pack(side=tk.LEFT, padx=5)
        
        self.clear_btn = tk.Button(control_frame, text="Clear Raster", command=self.clear_raster)
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        
        self.step_label = tk.Label(control_frame, text="Step: 0")
        self.step_label.pack(side=tk.RIGHT, padx=10)
        
        # ---- Bind events ----
        self.grid_canvas.bind("<Motion>", self.on_hover)
        self.grid_canvas.bind("<Button-1>", self.on_click)
        
        # ---- Precompute neuron positions (circle layout) ----
        self.n = self.net.n_total
        self.positions = self._layout_neurons()
        self.neuron_radius = 8
        
        # ---- Store canvas item IDs for circles ----
        self.circle_ids = []
        self._draw_neurons()
        
        # ---- Start periodic update ----
        self.update_gui()
    
    def _layout_neurons(self):
        """Place neurons in a circle."""
        angles = np.linspace(0, 2*np.pi, self.n, endpoint=False)
        radius = 200
        cx, cy = 250, 250
        x = cx + radius * np.cos(angles)
        y = cy + radius * np.sin(angles)
        return np.column_stack([x, y])
    
    def _draw_neurons(self):
        """Create the neuron circles on the canvas."""
        self.circle_ids = []
        for (x, y) in self.positions:
            cid = self.grid_canvas.create_oval(
                x - self.neuron_radius, y - self.neuron_radius,
                x + self.neuron_radius, y + self.neuron_radius,
                fill='blue', outline='white', width=1
            )
            self.circle_ids.append(cid)
    
    def toggle_pause(self):
        self.paused = not self.paused
        self.pause_btn.config(text="Resume" if self.paused else "Pause")
    
    def clear_raster(self):
        self.spike_history = []
        self.raster_canvas.delete("all")
    
    def on_hover(self, event):
        """Show neuron ID and potential in the window title."""
        # Find nearest neuron
        x, y = event.x, event.y
        min_dist = float('inf')
        nearest = -1
        for i, (nx, ny) in enumerate(self.positions):
            d = (x - nx)**2 + (y - ny)**2
            if d < min_dist:
                min_dist = d
                nearest = i
        if nearest >= 0 and min_dist < (self.neuron_radius*3)**2:
            v = self.net.V[nearest].numpy()
            typ = 'E' if nearest < 160 else 'I'
            self.root.title(f"Neuron {nearest} ({typ})  V={v:.3f}")
        else:
            self.root.title("Spiking Neural Network - Live Monitor")
    
    def on_click(self, event):
        """Show additional info on click (e.g., incoming/outgoing weights)."""
        x, y = event.x, event.y
        min_dist = float('inf')
        nearest = -1
        for i, (nx, ny) in enumerate(self.positions):
            d = (x - nx)**2 + (y - ny)**2
            if d < min_dist:
                min_dist = d
                nearest = i
        if nearest >= 0 and min_dist < (self.neuron_radius*3)**2:
            # Popup window with weight stats
            self.show_neuron_details(nearest)
    
    def show_neuron_details(self, neuron_id):
        """Create a popup with weight statistics for the neuron."""
        top = tk.Toplevel(self.root)
        top.title(f"Neuron {neuron_id} Details")
        top.geometry("300x200")
        
        incoming = self.net.W[:, neuron_id].numpy()
        outgoing = self.net.W[neuron_id, :].numpy()
        
        text = (f"Incoming weights:\n"
                f"  mean = {incoming.mean():.4f}\n"
                f"  std  = {incoming.std():.4f}\n"
                f"  min  = {incoming.min():.4f}\n"
                f"  max  = {incoming.max():.4f}\n\n"
                f"Outgoing weights:\n"
                f"  mean = {outgoing.mean():.4f}\n"
                f"  std  = {outgoing.std():.4f}\n"
                f"  min  = {outgoing.min():.4f}\n"
                f"  max  = {outgoing.max():.4f}")
        lbl = tk.Label(top, text=text, justify=tk.LEFT)
        lbl.pack(padx=10, pady=10)
    
    def update_gui(self):
        """Main update loop – called periodically."""
        if not self.running:
            return
        
        if not self.paused:
            # ---- Read current network state ----
            V = self.net.V.numpy()
            spikes = self.net.spikes.numpy()
            W = self.net.W.numpy()
            
            # ---- Update neuron colours ----
            # Map V to colour: blue (low) -> red (high)
            v_min, v_max = -1.0, 1.0
            norm_V = np.clip((V - v_min) / (v_max - v_min), 0, 1)
            for i, cid in enumerate(self.circle_ids):
                # Convert to RGB: blue (0,0,255) -> red (255,0,0)
                r = int(255 * norm_V[i])
                b = int(255 * (1 - norm_V[i]))
                g = 0
                color = f'#{r:02x}{g:02x}{b:02x}'
                self.grid_canvas.itemconfig(cid, fill=color)
            
            # ---- Flash for spikes ----
            spike_indices = np.where(spikes > 0.5)[0]
            for idx in spike_indices:
                # Flash white for a short time (we'll set a timer to revert)
                self.grid_canvas.itemconfig(self.circle_ids[idx], fill='white', outline='yellow')
                # Schedule revert after 50 ms
                self.root.after(50, lambda i=idx: self.grid_canvas.itemconfig(
                    self.circle_ids[i], fill='', outline='white'
                ))
            
            # ---- Update raster ----
            self.step_count += 1
            for nid in spike_indices:
                self.spike_history.append((self.step_count, nid))
            if len(self.spike_history) > self.history_len:
                self.spike_history = self.spike_history[-self.history_len:]
            
            # Redraw raster
            self.raster_canvas.delete("all")
            if self.spike_history:
                # Normalize times to fit canvas width
                max_t = max(t for t, _ in self.spike_history)
                min_t = max(0, max_t - self.history_len)
                width = self.raster_canvas.winfo_width() if self.raster_canvas.winfo_width() > 1 else 900
                height = self.raster_canvas.winfo_height() if self.raster_canvas.winfo_height() > 1 else 150
                # Draw spikes as dots
                for t, nid in self.spike_history:
                    if t < min_t:
                        continue
                    x = (t - min_t) / self.history_len * width
                    y = (nid / self.n) * height
                    self.raster_canvas.create_oval(x-1, y-1, x+1, y+1, fill='black', outline='black')
            
            # ---- Update heatmap (show a 20x20 block) ----
            sample = W[:20, :20]
            # Normalize to 0-255 for image
            sample_norm = (sample - (-2)) / (4) * 255  # range -2..2 -> 0..255
            sample_norm = np.clip(sample_norm, 0, 255).astype(np.uint8)
            img = Image.fromarray(sample_norm, mode='L')
            img = img.resize((200, 200), Image.Resampling.NEAREST)
            imgtk = ImageTk.PhotoImage(image=img)
            self.heatmap_label.config(image=imgtk)
            self.heatmap_label.image = imgtk  # keep reference
            
            # Update step label
            self.step_label.config(text=f"Step: {self.step_count}")
        
        # Schedule next update
        self.root.after(self.update_interval, self.update_gui)
    
    def start(self):
        """Start the Tkinter main loop."""
        self.root.mainloop()
    
    def stop(self):
        """Stop the visualizer."""
        self.running = False
        self.root.quit()

In [5]:
import tkinter as tk
from tkinter import ttk
import numpy as np
from PIL import Image, ImageTk
import threading
import time

class TkNetworkVisualizer:
    def __init__(self, network, history_len=200, update_interval_ms=100):
        """
        network: your SpikingNeuralNetwork instance
        history_len: number of recent spikes to keep in raster
        update_interval_ms: how often to refresh the GUI (milliseconds)
        """
        self.net = network
        self.history_len = history_len
        self.update_interval = update_interval_ms
        
        # Data storage
        self.spike_history = []          # list of (time_step, neuron_id)
        self.step_count = 0
        
        # State
        self.paused = False
        self.running = True
        
        # Build the Tk window
        self.root = tk.Tk()
        self.root.title("Spiking Neural Network - Live Monitor")
        self.root.geometry("1000x700")
        
        # Create frames
        top_frame = tk.Frame(self.root)
        top_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        
        left_frame = tk.Frame(top_frame)
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        right_frame = tk.Frame(top_frame)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        # ---- Neuron grid (Canvas) ----
        self.grid_canvas = tk.Canvas(left_frame, width=500, height=500, bg='black')
        self.grid_canvas.pack(fill=tk.BOTH, expand=True)
        
        # ---- Weight heatmap (Label with image) ----
        self.heatmap_label = tk.Label(right_frame, text="Weight Heatmap", bg='white')
        self.heatmap_label.pack(fill=tk.BOTH, expand=True)
        
        # ---- Raster (Canvas) ----
        self.raster_canvas = tk.Canvas(self.root, width=900, height=150, bg='white')
        self.raster_canvas.pack(side=tk.BOTTOM, fill=tk.X, padx=10, pady=10)
        
        # ---- Controls ----
        control_frame = tk.Frame(self.root)
        control_frame.pack(side=tk.BOTTOM, fill=tk.X, pady=5)
        
        self.pause_btn = tk.Button(control_frame, text="Pause", command=self.toggle_pause)
        self.pause_btn.pack(side=tk.LEFT, padx=5)
        
        self.speed_scale = tk.Scale(control_frame, from_=0.1, to=5.0, resolution=0.1,
                                    orient=tk.HORIZONTAL, label="Speed", length=150)
        self.speed_scale.set(1.0)
        self.speed_scale.pack(side=tk.LEFT, padx=5)
        
        self.clear_btn = tk.Button(control_frame, text="Clear Raster", command=self.clear_raster)
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        
        self.step_label = tk.Label(control_frame, text="Step: 0")
        self.step_label.pack(side=tk.RIGHT, padx=10)
        
        # ---- Bind events ----
        self.grid_canvas.bind("<Motion>", self.on_hover)
        self.grid_canvas.bind("<Button-1>", self.on_click)
        
        # ---- Precompute neuron positions (circle layout) ----
        self.n = self.net.n_total
        self.positions = self._layout_neurons()
        self.neuron_radius = 8
        
        # ---- Store canvas item IDs for circles ----
        self.circle_ids = []
        self._draw_neurons()
        
        # ---- Start periodic update ----
        self.update_gui()
    
    def _layout_neurons(self):
        """Place neurons in a circle."""
        angles = np.linspace(0, 2*np.pi, self.n, endpoint=False)
        radius = 200
        cx, cy = 250, 250
        x = cx + radius * np.cos(angles)
        y = cy + radius * np.sin(angles)
        return np.column_stack([x, y])
    
    def _draw_neurons(self):
        """Create the neuron circles on the canvas."""
        self.circle_ids = []
        for (x, y) in self.positions:
            cid = self.grid_canvas.create_oval(
                x - self.neuron_radius, y - self.neuron_radius,
                x + self.neuron_radius, y + self.neuron_radius,
                fill='blue', outline='white', width=1
            )
            self.circle_ids.append(cid)
    
    def toggle_pause(self):
        self.paused = not self.paused
        self.pause_btn.config(text="Resume" if self.paused else "Pause")
    
    def clear_raster(self):
        self.spike_history = []
        self.raster_canvas.delete("all")
    
    def on_hover(self, event):
        """Show neuron ID and potential in the window title."""
        # Find nearest neuron
        x, y = event.x, event.y
        min_dist = float('inf')
        nearest = -1
        for i, (nx, ny) in enumerate(self.positions):
            d = (x - nx)**2 + (y - ny)**2
            if d < min_dist:
                min_dist = d
                nearest = i
        if nearest >= 0 and min_dist < (self.neuron_radius*3)**2:
            v = self.net.V[nearest].numpy()
            typ = 'E' if nearest < 160 else 'I'
            self.root.title(f"Neuron {nearest} ({typ})  V={v:.3f}")
        else:
            self.root.title("Spiking Neural Network - Live Monitor")
    
    def on_click(self, event):
        """Show additional info on click (e.g., incoming/outgoing weights)."""
        x, y = event.x, event.y
        min_dist = float('inf')
        nearest = -1
        for i, (nx, ny) in enumerate(self.positions):
            d = (x - nx)**2 + (y - ny)**2
            if d < min_dist:
                min_dist = d
                nearest = i
        if nearest >= 0 and min_dist < (self.neuron_radius*3)**2:
            # Popup window with weight stats
            self.show_neuron_details(nearest)
    
    def show_neuron_details(self, neuron_id):
        """Create a popup with weight statistics for the neuron."""
        top = tk.Toplevel(self.root)
        top.title(f"Neuron {neuron_id} Details")
        top.geometry("300x200")
        
        incoming = self.net.W[:, neuron_id].numpy()
        outgoing = self.net.W[neuron_id, :].numpy()
        
        text = (f"Incoming weights:\n"
                f"  mean = {incoming.mean():.4f}\n"
                f"  std  = {incoming.std():.4f}\n"
                f"  min  = {incoming.min():.4f}\n"
                f"  max  = {incoming.max():.4f}\n\n"
                f"Outgoing weights:\n"
                f"  mean = {outgoing.mean():.4f}\n"
                f"  std  = {outgoing.std():.4f}\n"
                f"  min  = {outgoing.min():.4f}\n"
                f"  max  = {outgoing.max():.4f}")
        lbl = tk.Label(top, text=text, justify=tk.LEFT)
        lbl.pack(padx=10, pady=10)
    
    def update_gui(self):
        """Main update loop – called periodically."""
        if not self.running:
            return
        
        if not self.paused:
            # ---- Read current network state ----
            V = self.net.V.numpy()
            spikes = self.net.spikes.numpy()
            W = self.net.W.numpy()
            
            # ---- Update neuron colours ----
            # Map V to colour: blue (low) -> red (high)
            v_min, v_max = -1.0, 1.0
            norm_V = np.clip((V - v_min) / (v_max - v_min), 0, 1)
            for i, cid in enumerate(self.circle_ids):
                # Convert to RGB: blue (0,0,255) -> red (255,0,0)
                r = int(255 * norm_V[i])
                b = int(255 * (1 - norm_V[i]))
                g = 0
                color = f'#{r:02x}{g:02x}{b:02x}'
                self.grid_canvas.itemconfig(cid, fill=color)
            
            # ---- Flash for spikes ----
            spike_indices = np.where(spikes > 0.5)[0]
            for idx in spike_indices:
                # Flash white for a short time (we'll set a timer to revert)
                self.grid_canvas.itemconfig(self.circle_ids[idx], fill='white', outline='yellow')
                # Schedule revert after 50 ms
                self.root.after(50, lambda i=idx: self.grid_canvas.itemconfig(
                    self.circle_ids[i], fill='', outline='white'
                ))
            
            # ---- Update raster ----
            self.step_count += 1
            for nid in spike_indices:
                self.spike_history.append((self.step_count, nid))
            if len(self.spike_history) > self.history_len:
                self.spike_history = self.spike_history[-self.history_len:]
            
            # Redraw raster
            self.raster_canvas.delete("all")
            if self.spike_history:
                # Normalize times to fit canvas width
                max_t = max(t for t, _ in self.spike_history)
                min_t = max(0, max_t - self.history_len)
                width = self.raster_canvas.winfo_width() if self.raster_canvas.winfo_width() > 1 else 900
                height = self.raster_canvas.winfo_height() if self.raster_canvas.winfo_height() > 1 else 150
                # Draw spikes as dots
                for t, nid in self.spike_history:
                    if t < min_t:
                        continue
                    x = (t - min_t) / self.history_len * width
                    y = (nid / self.n) * height
                    self.raster_canvas.create_oval(x-1, y-1, x+1, y+1, fill='black', outline='black')
            
            # ---- Update heatmap (show a 20x20 block) ----
            sample = W[:20, :20]
            # Normalize to 0-255 for image
            sample_norm = (sample - (-2)) / (4) * 255  # range -2..2 -> 0..255
            sample_norm = np.clip(sample_norm, 0, 255).astype(np.uint8)
            img = Image.fromarray(sample_norm, mode='L')
            img = img.resize((200, 200), Image.Resampling.NEAREST)
            imgtk = ImageTk.PhotoImage(image=img)
            self.heatmap_label.config(image=imgtk)
            self.heatmap_label.image = imgtk  # keep reference
            
            # Update step label
            self.step_label.config(text=f"Step: {self.step_count}")
        
        # Schedule next update
        self.root.after(self.update_interval, self.update_gui)
    
    def start(self):
        """Start the Tkinter main loop."""
        self.root.mainloop()
    
    def stop(self):
        """Stop the visualizer."""
        self.running = False
        self.root.quit()

## Input Creation for HalfCheetah

In [6]:
@tf.function
def create_external_input(observation, reward_value):
    """
    HalfCheetah observations → neurons 0-16
    Reward → neuron 17
    """
    ext = tf.zeros(N_TOTAL, dtype=tf.float32)
    
    # Scale and input observations
    obs_indices = tf.constant(OBSERVATION_INPUT_IDS)
    obs_values = tf.cast(observation, tf.float32) * OBSERVATION_SCALING
    ext = tf.tensor_scatter_nd_update(
        ext, tf.expand_dims(obs_indices, 1), obs_values
    )
    
    # Reward input
    reward_index = tf.constant(REWARD_INPUT_ID)
    reward_update = tf.ones(1) * tf.cast(reward_value, tf.float32) * REWARD_SCALING
    ext = tf.tensor_scatter_nd_update(
        ext, tf.expand_dims(reward_index, 1), reward_update
    )
    
    return ext

# Test with dummy observation
dummy_obs = np.zeros(OBSERVATION_DIM)
test_ext = create_external_input(dummy_obs, 1.0)
print(f"Observation inputs active: {tf.reduce_sum(test_ext[:17]).numpy():.2f}")
print(f"Reward input active: {test_ext[17].numpy():.2f}")

Observation inputs active: 0.00
Reward input active: 0.00


## Neuron Dynamics

In [7]:
@tf.function
def update_neurons(
    V, W, bias, prev_spikes,
    external_input,
    adaptive_threshold,
    base_threshold
):
    weighted_sum = tf.linalg.matvec(W, prev_spikes)

    V_new = (
        0.95 * V
        + weighted_sum
        + bias
        + external_input
    )

    spikes_new = tf.cast(
        V_new >= adaptive_threshold,
        tf.float32
    )

    V_new = tf.where(
        spikes_new > 0,
        V_RESET,
        V_new
    )

    threshold_new = (
        base_threshold
        + (
            adaptive_threshold - base_threshold
        ) * ADAPTIVE_THRESHOLD_DECAY
        + spikes_new * ADAPTIVE_THRESHOLD_RISE
    )

    return V_new, spikes_new, threshold_new

## Weight Modification (Asymmetric LTP/LTD)

In [8]:
@tf.function
def modify_weights_asymmetric(W, eligibility, dopamine):
    """
    Asymmetric plasticity:
    - LTP: dopamine > 0 → fast strengthening
    - LTD: dopamine < 0 → slow weakening
    """
    weight_change = tf.zeros_like(W)
    
    if dopamine > 0:
        # Fast LTP
        weight_change = LEARNING_RATE_LTP * dopamine * eligibility
    elif dopamine < 0:
        # Slow LTD
        weight_change = -LEARNING_RATE_LTD * (-dopamine) * eligibility
    
    W_new = W + weight_change
    
    # Apply type constraints
    for j in range(N_TOTAL):
        if j in EXCITATORY_IDS:
            col = W_new[:, j]
            W_new = tf.tensor_scatter_nd_update(
                W_new,
                tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                tf.maximum(col, WEIGHT_MIN_EXC)
            )
        elif j in INHIBITORY_IDS:
            col = W_new[:, j]
            W_new = tf.tensor_scatter_nd_update(
                W_new,
                tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                tf.minimum(col, 0.0)
            )
    
    W_new = tf.clip_by_value(W_new, WEIGHT_MIN_INH, WEIGHT_MAX)
    W_new = tf.linalg.set_diag(W_new, tf.zeros(N_TOTAL))
    
    return W_new, weight_change

## Homeostasis Functions

In [9]:
@tf.function
def synaptic_scaling(
    W,
    spike_counts,
    total_steps,
    target_rate=TARGET_FIRING_RATE
):

    total_steps = tf.cast(
        tf.maximum(total_steps, 1),
        tf.float32
    )

    actual_rate = spike_counts / total_steps

    scaling = (
        target_rate /
        (actual_rate + 1e-6)
    )

    # Much safer than 0.1 -> 10
    scaling = tf.clip_by_value(
        scaling,
        0.5,
        2.0
    )

    W_scaled = (
        W * tf.reshape(scaling, (-1, 1))
    )

    W_scaled = tf.linalg.set_diag(
        W_scaled,
        tf.zeros(N_TOTAL)
    )

    return W_scaled
    
@tf.function
def weight_decay(W, decay_rate=WEIGHT_DECAY_RATE):
    """Gentle weight decay to prevent rigid patterns"""
    W_decayed = W * decay_rate
    W_decayed = tf.linalg.set_diag(W_decayed, tf.zeros(N_TOTAL))
    return W_decayed

## Neurogenesis Function

In [10]:
def neurogenesis(network):
    """Replace least active neurons with fresh random weights"""
    # Calculate firing rates
    total_steps = tf.cast(NEUROGENESIS_INTERVAL, tf.float32)
    firing_rates = network.spike_counter / total_steps
    
    # Create mask for protected neurons
    protected_mask = tf.constant(
        [1.0 if i in EXCLUDE_FROM_REPLACEMENT else 0.0 for i in range(N_TOTAL)],
        dtype=tf.float32
    )
    
    # Find candidates below threshold and not protected
    below_threshold = tf.cast(firing_rates < ACTIVITY_THRESHOLD, tf.float32)
    candidates = below_threshold * (1.0 - protected_mask)
    
    candidate_indices = tf.where(candidates > 0)[:, 0]
    candidate_rates = tf.gather(firing_rates, candidate_indices)
    
    # If not enough candidates, take from all non-protected neurons
    if tf.shape(candidate_indices)[0] < NEURONS_TO_REPLACE:
        all_nonprotected = tf.where(protected_mask == 0)[:, 0]
        all_rates = tf.gather(firing_rates, all_nonprotected)
        sorted_indices = tf.argsort(all_rates)
        neurons_to_replace = tf.gather(all_nonprotected, sorted_indices[:NEURONS_TO_REPLACE])
    else:
        sorted_indices = tf.argsort(candidate_rates)
        neurons_to_replace = tf.gather(candidate_indices, sorted_indices[:NEURONS_TO_REPLACE])
    
    # Reset weights for these neurons
    W_numpy = network.W.numpy()
    V_numpy = network.V.numpy()
    threshold_numpy = network.adaptive_threshold.numpy()
    
    replaced_neurons = []
    for neuron_id in neurons_to_replace.numpy():
        neuron_id = int(neuron_id)
        replaced_neurons.append(neuron_id)
        
        # Reset incoming connections
        if neuron_id in EXCITATORY_IDS:
            W_numpy[:, neuron_id] = np.abs(np.random.uniform(0.01, 0.1, N_TOTAL))
        else:
            W_numpy[:, neuron_id] = -np.abs(np.random.uniform(0.01, 0.1, N_TOTAL))
        
        # Reset outgoing connections
        if neuron_id in EXCITATORY_IDS:
            W_numpy[neuron_id, :] = np.abs(np.random.uniform(0.01, 0.1, N_TOTAL))
        else:
            W_numpy[neuron_id, :] = -np.abs(np.random.uniform(0.01, 0.1, N_TOTAL))
        
        # Remove self-connection
        W_numpy[neuron_id, neuron_id] = 0.0
        
        # Reset state
        V_numpy[neuron_id] = 0.0
        threshold_numpy[neuron_id] = network.base_threshold.numpy()[neuron_id]
        network.eligibility[neuron_id, :] = 0
        network.eligibility[:, neuron_id] = 0
        network.V[neuron_id] = 0
        network.spikes[neuron_id] = 0
        network.prev_spikes[neuron_id] = 0
    # Apply updates
    network.W.assign(tf.constant(W_numpy, dtype=tf.float32))
    network.V.assign(tf.constant(V_numpy, dtype=tf.float32))
    network.adaptive_threshold.assign(tf.constant(threshold_numpy, dtype=tf.float32))
    
    # Reset spike counter
    network.spike_counter.assign(tf.zeros(N_TOTAL))
    network.steps_since_neurogenesis.assign(0)
    
    # Update statistics
    network.neurons_replaced_count.assign_add(len(replaced_neurons))
    network.replacement_history.append(replaced_neurons)
    
    return replaced_neurons

In [11]:
def enforce_ei_constraints(W):

    W_new = W

    for j in EXCITATORY_IDS:

        col = W_new[:, j]

        W_new = tf.tensor_scatter_nd_update(
            W_new,
            tf.stack(
                [
                    tf.range(N_TOTAL),
                    tf.fill([N_TOTAL], j)
                ],
                axis=1
            ),
            tf.maximum(
                col,
                WEIGHT_MIN_EXC
            )
        )

    for j in INHIBITORY_IDS:

        col = W_new[:, j]

        W_new = tf.tensor_scatter_nd_update(
            W_new,
            tf.stack(
                [
                    tf.range(N_TOTAL),
                    tf.fill([N_TOTAL], j)
                ],
                axis=1
            ),
            tf.minimum(
                col,
                0.0
            )
        )

    # No self connections
    W_new = tf.linalg.set_diag(
        W_new,
        tf.zeros(N_TOTAL)
    )

    return W_new

## Action Decoding

In [12]:
def decode_actions(network, mode=ACTION_DECODING_MODE):
    """
    Decode network activity to HalfCheetah actions.
    
    Args:
        network: The spiking network
        mode: "rate" (firing rate) or "potential" (membrane potential)
    
    Returns:
        actions: Array of 6 action values in [-1, 1]
    """
    actions = []
    
    for action_idx in range(ACTION_DIM):
        start = action_idx * NEURONS_PER_ACTION
        end = start + NEURONS_PER_ACTION
        action_neurons = ACTION_OUTPUT_IDS[start:end]
        
        if mode == "rate":
            # Use recent firing rates
            rates = []
            for neuron_id in action_neurons:
                # Get rate from spike counter over the window
                rate = network.spike_counter[neuron_id].numpy() / ACTION_WINDOW
                rates.append(rate)
            
            avg_rate = np.mean(rates)
            
        elif mode == "potential":
            # Use current membrane potentials
            potentials = []
            for neuron_id in action_neurons:
                potentials.append(network.V[neuron_id].numpy())
            
            avg_potential = np.mean(potentials)
            avg_rate = avg_potential / V_THRESHOLD_EXC
            
        else:
            # Spike-based: count current spikes
            spikes = []
            for neuron_id in action_neurons:
                spikes.append(network.spikes[neuron_id].numpy())
            
            avg_rate = np.mean(spikes)
        
        # Scale to action range [-1, 1]
        action = np.tanh(avg_rate * ACTION_SCALING * 5)
        actions.append(action)
    
    return np.array(actions)

## Complete Step Function for MuJoCo

In [13]:
def step_mujoco(
    network,
    observation,
    reward_value,
    step_counter=0
):
    # --------------------------------------------------
    # INPUT
    # --------------------------------------------------

    ext_input = create_external_input(
        observation,
        reward_value
    )

    dopamine = 0.0

    # --------------------------------------------------
    # INTERNAL SNN STEPS
    # --------------------------------------------------

    for _ in range(INTERNAL_STEPS):

        V_new, spikes_new, threshold_new = update_neurons(
            network.V,
            network.W,
            network.bias,
            network.prev_spikes,
            ext_input,
            network.adaptive_threshold,
            network.base_threshold
        )

        # Activity tracking
        network.scaling_spike_counter.assign_add(spikes_new)
        network.neurogenesis_spike_counter.assign_add(spikes_new)

        network.scaling_steps.assign_add(1)
        network.neurogenesis_steps.assign_add(1)

        # State update
        network.V.assign(V_new)
        network.spikes.assign(spikes_new)
        network.prev_spikes.assign(spikes_new)
        network.adaptive_threshold.assign(threshold_new)

    # --------------------------------------------------
    # DOPAMINE / REWARD
    # --------------------------------------------------

    # Prediction error against OLD baseline
    dopamine = reward_value - network.baseline

    # Then update baseline
    network.baseline.assign(
        network.baseline
        + BASELINE_LEARNING_RATE * dopamine
    )

    # --------------------------------------------------
    # PLASTICITY
    # --------------------------------------------------

    spike_pair = tf.einsum(
        'i,j->ij',
        network.prev_spikes,
        network.prev_spikes
    )

    network.eligibility.assign(
        ELIGIBILITY_DECAY * network.eligibility
        + spike_pair
    )

    dw = (
        LEARNING_RATE
        * dopamine
        * network.eligibility
    )

    network.W.assign_add(dw)

    # Weight decay
    network.W.assign(
        network.W * (1.0 - WEIGHT_DECAY)
    )

    # --------------------------------------------------
    # E/I CONSTRAINTS
    # --------------------------------------------------

    network.W.assign(
        enforce_ei_constraints(network.W)
    )

    # --------------------------------------------------
    # PERIODIC SYNAPTIC SCALING
    # --------------------------------------------------

    if (
        int(network.scaling_steps.numpy())
        >= SYNAPTIC_SCALING_INTERVAL
    ):
        network.W.assign(
            synaptic_scaling(
                network.W,
                network.scaling_spike_counter,
                network.scaling_steps
            )
        )

        network.W.assign(
            enforce_ei_constraints(network.W)
        )

        network.scaling_spike_counter.assign(
            tf.zeros(N_TOTAL)
        )

        network.scaling_steps.assign(0)

    # --------------------------------------------------
    # PERIODIC NEUROGENESIS
    # --------------------------------------------------

    if (
        int(network.neurogenesis_steps.numpy())
        >= NEUROGENESIS_INTERVAL
    ):
        neurogenesis(network)

        network.neurogenesis_spike_counter.assign(
            tf.zeros(N_TOTAL)
        )

        network.neurogenesis_steps.assign(0)

    # --------------------------------------------------
    # ACTION
    # --------------------------------------------------

    action = decode_action(network)

    return action, {
        "dopamine": float(dopamine),
        "avg_firing_rate": float(
            tf.reduce_mean(
                network.scaling_spike_counter
                / tf.cast(
                    tf.maximum(network.scaling_steps, 1),
                    tf.float32
                )
            ).numpy()
        )
    }

### 2

## Exploration Noise

In [14]:
def add_exploration_noise(actions, exploration_rate=0.1):
    """Add Gaussian noise for exploration"""
    noise = np.random.normal(0, exploration_rate, size=actions.shape)
    noisy_actions = actions + noise
    return np.clip(noisy_actions, -1.0, 1.0)

## Main Training Loop

In [15]:
import threading
# Create environment
env = gym.make('Ant-v5',render_mode='human',ctrl_cost_weight=0.0, healthy_z_range=(0.5,1.0),healthy_reward=0)

# Initialize network
net = SpikingNeuralNetwork()

# Training parameters
N_EPISODES = 100
MAX_STEPS_PER_EPISODE = 1000

# History tracking
episode_rewards = []
episode_lengths = []
action_history = []
baseline_history = []
dopamine_history = []
neurogenesis_events = []

print("Starting HalfCheetah training with spiking network...")
print(f"Episodes: {N_EPISODES}")
print(f"Max steps per episode: {MAX_STEPS_PER_EPISODE}")
print(f"Internal steps per env step: {INTERNAL_STEPS}")
print()
# Create visualizer
vis = TkNetworkVisualizer(net)
# Start GUI in main thread (but we'll use after to start training)
def train_thread():
    # Your training loop here
    for episode in range(N_EPISODES):
        # Reset environment
        observation, info = env.reset()
        net.reset_state()
        
        total_reward = 0
        episode_actions = []
        episode_dopamine = []
        
        for step_num in range(MAX_STEPS_PER_EPISODE):
            # Get actions from network
            actions, net_info = step_mujoco(
                net, observation, total_reward, step_num
            )
            
            # Add exploration noise (more in early episodes)
            if episode < 30:
                # Use mostly random actions for initial exploration
                if np.random.random() < 0.7:  # 70% random actions
                    actions = np.random.uniform(-1, 1, ACTION_DIM)
                else:
                    actions = add_exploration_noise(actions, exploration_rate=0.5)
            elif episode < 50:
                # Moderate exploration
                actions = add_exploration_noise(actions, exploration_rate=0.3)
            else:
                # Less exploration
                actions = add_exploration_noise(actions, exploration_rate=0.1)
            
            # Execute action in environment
            observation, reward, terminated, truncated, info = env.step(actions)
            
            # Track
            total_reward += reward
            episode_actions.append(actions)
            episode_dopamine.append(net_info['dopamine'])
            
            if net_info['replaced_neurons']:
                neurogenesis_events.append((episode, step_num, net_info['replaced_neurons']))
            
            if terminated or truncated:
                break
        
        # Track episode stats
        episode_rewards.append(total_reward)
        episode_lengths.append(step_num + 1)
        action_history.append(np.array(episode_actions))
        baseline_history.append(net.baseline.numpy())
        dopamine_history.append(np.mean(episode_dopamine))
        
        if episode % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f"Episode {episode}: Reward = {total_reward:.1f}, "
                  f"Steps = {step_num+1}, Avg Reward (last 10) = {avg_reward:.1f}, "
                  f"Baseline = {net.baseline.numpy():.2f}, "
                  f"Active neurons = {net_info['active_neurons']}")
    env.close()

# Start training in a separate thread
threading.Thread(target=train_thread, daemon=True).start()
# Start GUI (blocks until window closed)
vis.start()

Starting HalfCheetah training with spiking network...
Episodes: 100
Max steps per episode: 1000
Internal steps per env step: 15



Exception in thread Thread-6 (train_thread):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2918/2971252849.py", line 41, in train_thread
    actions, net_info = step_mujoco(
                        ~~~~~~~~~~~^
        net, observation, total_reward, step_num
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/tmp/ipykernel_2918/959518704.py", line 76, in step_mujoco
    LEARNING_RATE
NameError: name 'LEARNING_RATE' is not defined. Did you mean: 'LEARNING_RATE_LTP'?


## Visualization of Training Results

## Performance Analysis

## Save Results

## Summary

This notebook implements a dopamine-modulated spiking neural network for HalfCheetah control with:

1. **Input**: 17 observations → neurons 0-16, reward → neuron 17
2. **Processing**: 64 neurons with all-to-all connectivity (51 excitatory, 13 inhibitory)
3. **Output**: 6 actions decoded from firing rates of neurons 18-35
4. **Learning**: Dopamine baseline (reward prediction error) with asymmetric LTP/LTD
5. **Homeostasis**: Adaptive thresholds, synaptic scaling, weight decay
6. **Neurogenesis**: Replaces inactive neurons every 500 steps

The network runs 10 internal steps per environment step for richer dynamics.